In [0]:
%sql
-- DISEÑO DE TABLA PARA LA CAPA SILVER
DROP VIEW IF EXISTS ventas_bronze_EDA;
CREATE OR REPLACE VIEW ventas_bronze_EDA AS

WITH limpieza AS (
    SELECT
        venta,
        articulo,

        -- Limpieza de descripción
        TRIM(
            regexp_replace(
                regexp_replace(
                    regexp_replace(lower(descrip), '[\".]', ' '),
                    '[()]',
                    ''
                ),
                '\\s+',
                ' '
            )
        ) AS clean_descrip,

        TRY_CAST(REGEXP_REPLACE(precio, ',', '.') AS DOUBLE) AS precio,
        TRY_CAST(cant AS INT) AS cant,
        TRY_CAST(REGEXP_REPLACE(total, ',', '.') AS DOUBLE) AS total,

        LOWER(TRIM(usulogin)) AS usulogin,
        LOWER(TRIM(clinombre)) AS clinombre,
        cliente,
        vtaoperacion,
        LOWER(TRIM(vtaestado)) AS vtaestado,
        TO_TIMESTAMP(vtafecha, 'd/M/yyyy HH:mm') AS vtafecha,
        condvtapos,
        LOWER(TRIM(delivery)) AS delivery,
        turno,
        caja,
        comprobante,
        LOWER(TRIM(succodigo)) AS succodigo,
        sucursal   

    FROM catalog_ventas.raw.ventas_heladeria_bronze
)

SELECT
    venta,
    articulo,

    -- Estandarización + eliminación de "envios a domicilio"
    CASE 
        WHEN clean_descrip LIKE '%1/2 kilo%' THEN '1/2kilo'
        WHEN clean_descrip LIKE '%1 kilo%' THEN '1kilo'
        WHEN clean_descrip LIKE '%1/4 kilo%' THEN '1/4kilo'
        WHEN clean_descrip LIKE '%sabores%' THEN '1/4kilo'
        ELSE 
        NULLIF(
            TRIM(
                regexp_replace(
                    regexp_replace(
                        clean_descrip,
                        'envios? a domicilio|envio|domicilio|delivery',
                        ''
                    ),
                    '\\s+',
                    ' '
                )
            ),
            ''
        )
    END AS descrip,

    precio,
    cant,
    total,
    usulogin,
    clinombre,
    cliente,
    vtaoperacion,
    vtaestado,
    vtafecha,
    condvtapos,
    delivery,
    turno,
    caja,
    comprobante,
    succodigo
FROM limpieza;

In [0]:
%sql
-- VERIFICAR LOS DATOS
SELECT *
FROM ventas_bronze_EDA LIMIT 5

In [0]:
%sql
-- VERIFICAR ESTRUCTURA
DESCRIBE ventas_bronze_eda